# 02 · Branch-and-Bound with PuLP
**Branch-and-Cut Workshop · UANL & Tecnológico de Monterrey** (Prof. Angélica Salazar, Prof. Vincent Boyer)

*Activity 3 · about 8 min.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you. Cells marked **🤖 AI prompt** contain a prompt you can paste into
> Gemini (the ✨ button in Colab), ChatGPT, Claude… **Always read and test the code the AI gives you.**

**Idea.** The LP relaxation gives an **upper bound** (max problem). If its solution is fractional, split the
problem in two on a fractional variable $x_j = v$:
$\;x_j \le \lfloor v \rfloor\;$ or $\;x_j \ge \lfloor v \rfloor + 1$. No integer point is lost, and the
current fractional point is removed from both children.

A node is **pruned** (not explored further) when its LP is
* **infeasible**,
* **integer**: a candidate solution (the best one so far is the *incumbent*),
* **dominated**: its LP bound is $\le$ the incumbent, so nothing better can be found below it.

PuLP solves the LP of each node; the logic of the tree is ours.

In [ ]:
# @title Setup and helper functions (run this cell; no need to read it)
# PuLP 4 with the open-source HiGHS solver (not pre-installed on Colab: installed on first run, ~20 s)
try:
    import pulp, highspy
    if int(pulp.__version__.split(".")[0]) < 4:
        raise ImportError("old PuLP")
except ImportError:
    import importlib, subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[highs]>=4"], check=True)
    for _m in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[_m]                       # forget an older PuLP loaded earlier in this session
    importlib.invalidate_caches()
    import pulp
SOLVER = pulp.HiGHS(msg=False)                    # every model in these notebooks is solved with HiGHS
print("PuLP", pulp.__version__, "+ HiGHS")


import math, re
from fractions import Fraction as Fr
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown

def lp_data(prob):
    """Read  max c x  s.t.  A x <= b,  x >= 0  from a PuLP model, in exact fractions.
    '>=' rows are multiplied by -1; a 'min' objective is turned into 'max -c x'."""
    xs = prob.variables()
    if (not xs or not prob.objective or all(prob.objective.get(v, 0) == 0 for v in xs)
            or all(con.get(v, 0) == 0 for con in prob.constraints() for v in xs)):
        raise ValueError(f"Model '{prob.name}' has no variables or no constraints with non-zero coefficients. "
                         "Did you fill in the ✍️ TODO lines (objective and constraints)?")
    q = lambda v: Fr(v).limit_denominator(10**6)
    sign = 1 if prob.sense == pulp.LpMaximize else -1
    c = [sign * q(prob.objective.get(v, 0)) for v in xs]
    A, b = [], []
    for con in prob.constraints():
        row, rhs = [q(con.get(v, 0)) for v in xs], -q(con.constant)
        if con.sense == pulp.LpConstraintGE:
            row, rhs = [-a for a in row], -rhs
        elif con.sense == pulp.LpConstraintEQ:
            raise ValueError("this small demo handles <= and >= constraints only")
        A.append(row); b.append(rhs)
    return [v.name for v in xs], c, A, b

def matmul(X, Y):
    return [[sum(X[i][k] * Y[k][j] for k in range(len(Y))) for j in range(len(Y[0]))] for i in range(len(X))]

def nice(v, den=1000):
    """Float from the solver -> short fraction string, e.g. 3.3333333 -> '10/3'."""
    return str(Fr(v).limit_denominator(den))

# ---- LaTeX output (fractions shown as fractions) ----
def tex_num(q):
    q = Fr(q)
    if q.denominator == 1:
        return str(q.numerator)
    s = r"\tfrac{%d}{%d}" % (abs(q.numerator), q.denominator)
    return "-" + s if q < 0 else s

def tex_var(name):
    return re.sub(r"^([A-Za-z]+)_?(\d+)$", r"\1_{\2}", name)

def tex_mat(M):
    return r"\begin{pmatrix}" + r"\\ ".join(" & ".join(tex_num(v) for v in row) for row in M) + r"\end{pmatrix}"

def tex_col(v):
    return tex_mat([[a] for a in v])

def tex_expr(coefs, names):
    out = ""
    for a, nm in zip(coefs, names):
        a = Fr(a)
        if a == 0:
            continue
        sign = ("-" if a < 0 else "") if not out else (" - " if a < 0 else " + ")
        out += sign + ("" if abs(a) == 1 else tex_num(abs(a)) + " ") + tex_var(nm)
    return out or "0"

def show(*parts):
    display(Math(r"\qquad ".join(parts)))

def plot_model(prob, path=(), cuts=(), points=True, star=None, ax=None, title=""):
    """Feasible region of a 2-variable model, its integer points, a simplex path and cuts (a, beta, label)."""
    names, c, A, b = lp_data(prob)
    A, b = np.array(A, float), np.array(b, float)
    hi = 1.15 * max(b[i] / A[i, j] for i in range(len(b)) for j in range(2) if A[i, j] > 0)
    ax = ax or plt.subplots(figsize=(5.5, 4.5))[1]
    X, Y = np.meshgrid(np.linspace(0, hi, 500), np.linspace(0, hi, 500))
    ok = np.all([A[i, 0] * X + A[i, 1] * Y <= b[i] + 1e-9 for i in range(len(b))], axis=0)
    ax.contourf(X, Y, ok, levels=[0.5, 1], colors=["#0072CE"], alpha=0.15)
    xs = np.linspace(0, hi, 200)
    for i in range(len(b)):
        if A[i, 1] != 0:
            ax.plot(xs, (b[i] - A[i, 0] * xs) / A[i, 1], color="#003865", lw=1)
    if points:
        pts = [(i, j) for i in range(int(hi) + 1) for j in range(int(hi) + 1)
               if all(A[k, 0] * i + A[k, 1] * j <= b[k] + 1e-9 for k in range(len(b)))]
        ax.scatter(*zip(*pts), s=14, color="k", zorder=3)
    for k, (a, beta, label) in enumerate(cuts):
        a = [float(v) for v in a]
        ax.plot(xs, (float(beta) - a[0] * xs) / a[1], "--", lw=2, label=label)
    if len(path):
        P = np.array([[float(v) for v in p] for p in path])
        ax.plot(P[:, 0], P[:, 1], "-o", color="#E07B00", lw=2, zorder=4, label="simplex path")
    if star is not None:
        ax.scatter([float(star[0])], [float(star[1])], marker="*", s=220, color="#C8102E", zorder=5)
    ax.set_xlim(-0.2, hi); ax.set_ylim(-0.2, hi); ax.set_xlabel(names[0]); ax.set_ylabel(names[1])
    ax.set_title(title or prob.name)
    if cuts or len(path):
        ax.legend(fontsize=8, loc="upper right")
    return ax

# E: the explanation problem (used on the slides)
def make_E(integer=False):
    cat = "Integer" if integer else "Continuous"
    E = pulp.LpProblem("E", pulp.LpMaximize)
    x1 = E.add_variable("x1", lowBound=0, cat=cat)
    x2 = E.add_variable("x2", lowBound=0, cat=cat)
    E += 5 * x1 + 8 * x2                    # objective
    E += x1 + x2 <= 6, "c1"
    E += 5 * x1 + 9 * x2 <= 45, "c2"
    return E

# P: the practice problem
def make_P(integer=False):
    cat = "Integer" if integer else "Continuous"
    P = pulp.LpProblem("P", pulp.LpMaximize)
    x1 = P.add_variable("x1", lowBound=0, cat=cat)
    x2 = P.add_variable("x2", lowBound=0, cat=cat)
    P += 3 * x1 + 4 * x2
    P += x1 + 2 * x2 <= 8, "c1"
    P += 2 * x1 + x2 <= 9, "c2"
    return P

## Solving one node
`solve_node(model, bounds)` solves the LP relaxation with extra bounds on the variables.

In [ ]:
def solve_node(prob, bounds):
    """LP relaxation of prob with extra bounds, e.g. {"x1": (None, 2)} means x1 <= 2.
    Returns (x, z) with x a dict {name: value}, or (None, None) if the node LP is infeasible."""
    saved = {v.name: (v.lowBound, v.upBound, v.cat) for v in prob.variables()}
    try:
        for v in prob.variables():
            v.cat = pulp.LpContinuous                      # relax integrality
            lo, up = bounds.get(v.name, (None, None))
            if lo is not None:
                v.lowBound = lo
            if up is not None:
                v.upBound = up
        stats = prob.solve(SOLVER)
        if stats.status != pulp.LpSolveStatus.Optimal:
            return None, None
        return {v.name: v.value() for v in prob.variables()}, pulp.value(prob.objective)
    finally:                                               # restore the model as it was
        for v in prob.variables():
            v.lowBound, v.upBound, v.cat = saved[v.name]

def branch_and_bound(prob, left_first=True, verbose=True):
    """Depth-first B&B: branch on the first fractional variable. Returns (best_x, best_z, nodes)."""
    nodes, best = [], {"x": None, "z": -math.inf}

    def explore(bounds, parent, label, depth):
        x, z = solve_node(prob, bounds)
        k = len(nodes)
        node = dict(id=k, parent=parent, label=label, x=x, z=z, depth=depth, bounds=bounds)
        nodes.append(node)
        kids = []
        if x is None:
            node["status"] = "infeasible"
        elif z <= best["z"] + 1e-9:
            node["status"] = "pruned by bound"
        else:
            frac = [nm for nm, v in x.items() if abs(v - round(v)) > 1e-6]
            if not frac:
                node["status"] = "integer"
                best.update(x=x, z=z)
            else:
                node["status"] = "branch"
                nm = frac[0]; f = math.floor(x[nm]); lo, up = bounds.get(nm, (None, None))
                kids = [({**bounds, nm: (lo, f)}, f"{nm} ≤ {f}"), ({**bounds, nm: (f + 1, up)}, f"{nm} ≥ {f + 1}")]
                if not left_first:
                    kids.reverse()
        if verbose:
            sol = "" if x is None else "x = (" + ", ".join(nice(v) for v in x.values()) + f"), z = {nice(z)}"
            print("    " * depth + f"[{k}] {label:<8} {sol:<28} -> {node['status']}"
                  + ("  (new incumbent)" if node["status"] == "integer" else ""))
        for b2, lab in kids:
            explore(b2, k, lab, depth + 1)

    explore({}, None, "root", 0)
    best["nodes"] = nodes
    return best["x"], best["z"], nodes

def draw_tree(nodes, title=""):
    """Draw the B&B tree: blue = branched, green = integer, red = pruned, grey = infeasible."""
    kids = {n["id"]: [m["id"] for m in nodes if m["parent"] == n["id"]] for n in nodes}
    pos, leaf = {}, [0]
    def place(k):
        if not kids[k]:
            pos[k] = leaf[0]; leaf[0] += 1
        else:
            for c in kids[k]:
                place(c)
            pos[k] = sum(pos[c] for c in kids[k]) / len(kids[k])
    place(0)
    color = {"branch": "#D0E8F8", "integer": "#B8E0C8", "pruned by bound": "#F5C6CB", "infeasible": "#DDDDDD"}
    depth = max(n["depth"] for n in nodes)
    fig, ax = plt.subplots(figsize=(max(5, 2.3 * leaf[0]), 1.5 * (depth + 1)))
    zbest = max((n["z"] for n in nodes if n["status"] == "integer"), default=None)
    for n in nodes:
        X, Y = pos[n["id"]], -n["depth"]
        if n["parent"] is not None:
            ax.plot([pos[n["parent"]], X], [-(n["depth"] - 1), Y], color="gray", lw=1, zorder=1)
        txt = f"[{n['id']}] {n['label']}\n" + ("infeasible" if n["x"] is None else
               "(" + ", ".join(nice(v) for v in n["x"].values()) + f")\nz = {nice(n['z'])}")
        bold = n["status"] == "integer" and n["z"] == zbest
        ax.text(X, Y, txt, ha="center", va="center", fontsize=8, zorder=2,
                bbox=dict(boxstyle="round", fc=color[n["status"]], ec="black", lw=2 if bold else 0.6))
    ax.set_xlim(-0.7, leaf[0] - 0.3); ax.set_ylim(-depth - 0.6, 0.6); ax.axis("off")
    ax.set_title(title or f"{len(nodes)} nodes"); plt.show()

In [ ]:
E = make_E(integer=True)
print("root     :", solve_node(E, {}))
print("x1 <= 2  :", solve_node(E, {"x1": (None, 2)}))
print("x1 >= 3  :", solve_node(E, {"x1": (3, None)}))

## The whole tree for $E$ (live demo)
Rule: branch on the **first** fractional variable; explore the **left** child ($\le$) first (depth-first).

In [ ]:
x_best, z_best, nodes = branch_and_bound(E)
print("\noptimum:", x_best, "z =", z_best, "| nodes:", len(nodes))
draw_tree(nodes, "E: Branch-and-Bound tree")

**Things to notice.** The incumbent improves $34 \to 37 \to 40$. The right child of the root ($x_1 \ge 3$, $z = 39$)
is explored last and **pruned by bound**: its LP value is below the incumbent 40.

---
## ✍️ Activity 3 — the tree of $P$
1. **Before running anything:** the root LP of $P$ is $(\tfrac{10}3, \tfrac73)$ (Activity 2). Which variable do
   you branch on, and what are the two children?
2. Solve the two children with `solve_node` and decide for each one: *branch*, *integer*, *infeasible* or
   *pruned by bound*. Continue until no open node is left.
3. **Post on Miro** (frame *3 – B&B tree*): **your own drawing** of the tree (paper photo or Miro shapes), and
   **why** each leaf was closed.
4. **Only then**, check yourself: uncomment the cell after the next one and run it.

In [ ]:
P = make_P(integer=True)
print("root    :", solve_node(P, {}))
# ✍️ TODO: the two children of the root, then their children if needed, e.g.
# print("x1 <= 3 :", solve_node(P, {"x1": (None, 3)}))

In [ ]:
# ✅ Check AFTER posting your tree on Miro: remove the two # below, then run this cell.
# x_best, z_best, nodes = branch_and_bound(P)
# draw_tree(nodes, "P: Branch-and-Bound tree")

## 🚀 If you have time
* Run `branch_and_bound(P, left_first=False)`. Same optimum? Same number of nodes? Why can the order matter?
* Add the inequality $x_1 + x_2 \le 5$ to $P$ (`P += P.variables()[0] + P.variables()[1] <= 5`) and run B&B again.
  How many nodes now? Is the inequality satisfied by every integer point of $P$? → Part 4.